# Playwright Básico: Locators y Aserciones

## Introducción

Playwright es una herramienta moderna para automatización de navegadores que permite escribir pruebas E2E confiables y rápidas. En este notebook aprenderemos los conceptos fundamentales.

---

## 1. Locators en Playwright

Los locators son la forma en que Playwright encuentra elementos en la página. Los locators más robustos usan `data-testid`.

In [ ]:
from playwright.sync_api import sync_playwright

# Ejemplo de diferentes tipos de locators

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto('http://localhost:5000')
    
    # Locator por data-testid (RECOMENDADO)
    title_input = page.locator('[data-testid="task-title-input"]')
    
    # Locator por texto
    create_button = page.get_by_text('Agregar Tarea')
    
    # Locator por rol
    submit_button = page.get_by_role('button', name='Agregar Tarea')
    
    # Locator por placeholder
    title_input_placeholder = page.get_by_placeholder('Título de la tarea')
    
    browser.close()

print("Locators demostrados exitosamente")

## 2. Por qué usar data-testid

### ❌ Locators frágiles (evitar)
```python
# XPath frágil - se rompe si cambia el DOM
page.locator('xpath=/html/body/div[1]/form/input[1]')

# Selector CSS frágil - se rompe si cambia la estructura
page.locator('div.container > form > input:first-child')

# Selector por clase - se rompe si cambia el CSS
page.locator('.task-form input')
```

### ✅ Locators robustos (usar)
```python
# Por data-testid - estable ante cambios de CSS/DOM
page.locator('[data-testid="task-title-input"]')

# Por texto semántico - estable ante cambios visuales
page.get_by_text('Agregar Tarea')

# Por rol - accesible y semántico
page.get_by_role('button', name='Agregar Tarea')
```

## 3. Aserciones en Playwright

Playwright incluye aserciones específicas para E2E que esperan condiciones antes de fallar.

In [ ]:
from playwright.sync_api import sync_playwright, expect

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    page.goto('http://localhost:5000')
    
    # Ejemplo de aserciones
    
    # Verificar que un elemento es visible
    # expect(page.locator('[data-testid="task-form"]')).to_be_visible()
    
    # Verificar que un elemento contiene texto
    # expect(page.locator('[data-testid="page-title"]')).to_contain_text('Gestor de Tareas')
    
    # Verificar que un elemento tiene cierto atributo
    # expect(page.locator('[data-testid="task-title-input"]')).to_have_attribute('type', 'text')
    
    # Verificar que un elemento está habilitado
    # expect(page.locator('[data-testid="create-task-button"]')).to_be_enabled()
    
    browser.close()

print("Aserciones demostradas exitosamente")

## 4. Ejemplo: Prueba fuerte de creación de tarea

Esta prueba verifica realmente que la tarea se crea y aparece en la lista.

In [ ]:
from playwright.sync_api import sync_playwright, expect

def test_create_task_strong():
    """Prueba fuerte que verifica que la tarea realmente se crea."""
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto('http://localhost:5000')
        
        # Llenar el formulario
        page.fill('[data-testid="task-title-input"]', 'Tarea de prueba')
        page.fill('[data-testid="task-description-input"]', 'Descripción de prueba')
        
        # Enviar el formulario
        page.click('[data-testid="create-task-button"]')
        
        # VERIFICACIÓN FUERTE: La tarea debe aparecer en la lista
        task_items = page.locator('[data-testid="task-item"]')
        expect(task_items).to_have_count(1)
        
        # Verificar que el título es correcto
        task_title = page.locator('[data-testid="task-title"]')
        expect(task_title).to_contain_text('Tarea de prueba')
        
        # Verificar que la descripción aparece
        task_description = page.locator('[data-testid="task-description"]')
        expect(task_description).to_contain_text('Descripción de prueba')
        
        browser.close()

# test_create_task_strong()
print("Prueba fuerte de creación definida")

## 5. Ejemplo: Prueba fuerte de completar tarea

Esta prueba verifica que al completar una tarea, aparece el badge correspondiente.

In [ ]:
def test_complete_task_strong():
    """Prueba fuerte que verifica que la tarea se marca como completada."""
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto('http://localhost:5000')
        
        # Crear una tarea primero
        page.fill('[data-testid="task-title-input"]', 'Tarea para completar')
        page.click('[data-testid="create-task-button"]')
        
        # Verificar que la tarea no está completada
        completed_badge = page.locator('[data-testid="completed-badge"]')
        expect(completed_badge).to_have_count(0)
        
        # Completar la tarea
        page.click('[data-testid="complete-button"]')
        
        # VERIFICACIÓN FUERTE: El badge debe aparecer
        expect(completed_badge).to_have_count(1)
        expect(completed_badge).to_contain_text('✓ Completada')
        
        # Verificar que el elemento tiene la clase completed
        task_item = page.locator('[data-testid="task-item"]')
        expect(task_item).to_have_class(/completed/)
        
        browser.close()

# test_complete_task_strong()
print("Prueba fuerte de completación definida")

## 6. Ejemplo: Prueba fuerte de eliminar tarea

Esta prueba verifica que al eliminar una tarea, desaparece de la lista.

In [ ]:
def test_delete_task_strong():
    """Prueba fuerte que verifica que la tarea se elimina correctamente."""
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto('http://localhost:5000')
        
        # Crear una tarea primero
        page.fill('[data-testid="task-title-input"]', 'Tarea para eliminar')
        page.click('[data-testid="create-task-button"]')
        
        # Verificar que hay una tarea
        task_items = page.locator('[data-testid="task-item"]')
        expect(task_items).to_have_count(1)
        
        # Eliminar la tarea
        page.click('[data-testid="delete-button"]')
        
        # VERIFICACIÓN FUERTE: La tarea debe desaparecer
        expect(task_items).to_have_count(0)
        
        # Verificar que aparece el estado vacío
        empty_state = page.locator('[data-testid="empty-state"]')
        expect(empty_state).to_be_visible()
        expect(empty_state).to_contain_text('No hay tareas')
        
        browser.close()

# test_delete_task_strong()
print("Prueba fuerte de eliminación definida")

## 7. Diferencia entre pruebas débiles y fuertes

### Prueba Débil (como las iniciales)
```python
def test_create_task_weak(page):
    page.fill('[data-testid="task-title-input"]', 'Test')
    page.click('[data-testid="create-task-button"]')
    # Solo verifica que no hubo error - NO verifica que se creó
    assert page.url == 'http://localhost:5000/'
```

### Prueba Fuerte (como las que debemos escribir)
```python
def test_create_task_strong(page):
    page.fill('[data-testid="task-title-input"]', 'Test')
    page.click('[data-testid="create-task-button"]')
    # VERIFICA que la tarea realmente aparece en la lista
    task_items = page.locator('[data-testid="task-item"]')
    expect(task_items).to_have_count(1)
    expect(page.locator('[data-testid="task-title"]')).to_contain_text('Test')
```

---

## Ejercicio

Implementa estas tres pruebas fuertes en `tests/test_tareas_e2e.py` bajo la clase `TestCrearTareaFuerte`.